<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/ai-act-conformity/lessons/P01-L06-accuracy-robustness-security/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/ai-act-conformity/lessons/P01-L06-accuracy-robustness-security/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/ai-act-conformity/lessons/P01-L06-accuracy-robustness-security/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/ai-act-conformity/lessons/P01-L06-accuracy-robustness-security/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P01-L06 · Accuracy, robustness and cybersecurity evidence

**You will build:** an Article 15 evidence report in which every measured number carries its
uncertainty — a metric with a bootstrap interval, a robustness degradation curve with the
severity at which the declared floor gives way, a poisoning scan, a measured model-extraction
budget, and a residual-risk statement that says only what the evidence bounds.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, no download, no network
· **Prerequisites:** `T10-L01-ai-act-conformity-pack`, `P01-L01-article-12-logging`

Article 15 asks for an appropriate level of accuracy, robustness and cybersecurity, and for
consistent performance in those respects throughout the lifecycle. Article 15(3) then asks
for the accuracy metrics to be declared in the instructions for use. Almost every pack you
will read answers that with a single bare number and no interval around it. A number with
no interval can be neither falsified nor defended, which is precisely why it is the one
everybody writes.

By the end you will be able to:

1. Implement a paired percentile bootstrap, and measure what happens to an interval when the
   predictions are resampled independently of the labels.
2. Implement a declaration that refuses a bare point estimate, and declare the bound the
   evidence supports rather than the point in the middle of it.
3. Seal an accuracy floor whose ordering the code enforces, so a floor chosen after the
   results is a `FloorAfterTheFactError` rather than a matter of professional judgement.
4. Measure a four-family degradation curve and report the severity at which the declared
   floor's lower bound gives way — which is not the severity at which the point does.
5. Implement a neighbourhood poisoning scan with a minimum-support rule and a measured
   extraction budget, and emit only the residual-risk statements the evidence bounds.

> **This is engineering, not legal advice.** The article numbers, the quoted wording and the
> dates are sourced in `claims.yaml` with their URLs and access dates. Every threshold below
> — the 0.86 accuracy floor, the 30-row support floor, the 15-neighbour window, the 0.95
> extraction target, the 90-day target — is **this lesson's modelling choice**, argued for
> where it appears. None of them is a statement about what any authority would accept. For a
> real system, read the Official Journal text and take professional advice.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import hashlib
import io
import json
import math
import sys
import traceback
from datetime import date
from typing import Any, Callable

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__)

LESSON_ID = "P01-L06"
SYSTEM_ID = "loan-copilot"          # the same fictional system P01-L01 logs and P01-L04 audits
AS_OF = date(2026, 9, 16)           # the date this evidence was produced. Fixed, so it re-runs.

print(f"{LESSON_ID} · system {SYSTEM_ID} · as of {AS_OF.isoformat()}")

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("bootstrap_ci",),
    "exercise 2": ("declare",),
    "exercise 3": ("seal_floor",),
    "exercise 4": ("perturbation_curve",),
    "exercise 5": ("crossing_severity",),
    "exercise 6": ("poison_scan",),
    "exercise 7": ("extraction_risk",),
    "exercise 8": ("residual_risk",),
    "exercise 9": ("unbounded_numbers",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 7"] -> "exercise 7 (extraction_risk)"; several -> "exercises 1, 3 and 5"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the other eight. A demo names the exercises it
    `needs`: until each has passed its check, the demo says which one it is waiting for and
    skips. Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board
    at the foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script
    run non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def canonical_bytes(obj: Any) -> bytes:
    """One deterministic serialisation of a JSON-shaped object. Given to you; not graded.

    The same convention P01-L01 used for its log: sorted keys, no insignificant whitespace.
    Hashing a dict is hashing ONE serialisation of it, so the serialisation is fixed here.
    """
    return json.dumps(obj, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=True, default=str).encode("utf-8")


def digest_of(obj: Any) -> str:
    """SHA-256 over `canonical_bytes(obj)`, as hex. Given to you; not graded."""
    return hashlib.sha256(canonical_bytes(obj)).hexdigest()


class UndeclarableMetricError(ValueError):
    """Raised when something is offered for declaration without a usable interval."""


class FloorAfterTheFactError(RuntimeError):
    """Raised when a performance floor is sealed after the thing it governs was measured."""


class TamperedFloorError(RuntimeError):
    """Raised when a sealed floor's digest does not recompute from its own contents."""

## 1. What Article 15 asks for

Four sentences do the work, and each one becomes a thing you compute in this lesson. They are
quoted verbatim against two independent reproductions of the consolidated text in
`claims.yaml`. Regulation (EU) 2026/1744 — the Digital Omnibus that deleted Article 10(5),
which `P01-L04` had to account for — moved the Chapter III application dates but left the
text of Article 15 alone.

In [ ]:
DUTIES = {
    "art15_1_levels": ("Article 15(1)",
                       "appropriate level of accuracy, robustness and cybersecurity, performing "
                       "consistently throughout the lifecycle"),
    "art15_3_declared": ("Article 15(3)",
                         "the levels of accuracy and the relevant accuracy metrics shall be "
                         "declared in the accompanying instructions of use"),
    "art15_4_resilience": ("Article 15(4)",
                           "as resilient as possible regarding errors, faults or inconsistencies "
                           "within the system or its environment"),
    "art15_5_security": ("Article 15(5)",
                         "resilient against unauthorised third parties; measures against data "
                         "poisoning, model poisoning, adversarial examples, confidentiality "
                         "attacks and model flaws"),
    "annex_iv_2g": ("Annex IV(2)(g)",
                    "metrics used to measure accuracy, robustness and compliance, and the test "
                    "logs and reports"),
}
print(f"{'duty':22s} {'article':18s} what it requires")
for _key, (_article, _what) in DUTIES.items():
    print(f"{_key:22s} {_article:18s} {_what[:70]}")
print(f"\n{len(DUTIES)} duties. Note which word Article 15(1) does NOT contain: a number. It "
      "asks for an\nAPPROPRIATE level and says nothing about what appropriate is — so every "
      "threshold in this\nlesson is a choice somebody has to be able to defend.")

### The part of 15(1) people skip

"Perform consistently in those respects throughout their lifecycle." A single accuracy figure
measured once on a clean test set says nothing about consistency. Robustness under
perturbation, a poisoning scan over the training set and an extraction budget are the three
ways this lesson turns "consistently" into things you can re-run next quarter and diff.

In [ ]:
OMNIBUS = {
    "2026-07-27": "Regulation (EU) 2026/1744 entered into force",
    "2027-12-02": "Chapter III applies to Annex III stand-alone high-risk systems",
    "2028-08-02": "Chapter III applies to Annex I product-embedded high-risk systems",
}
for _iso, _what in OMNIBUS.items():
    _state = "PAST" if date.fromisoformat(_iso) <= AS_OF else "-> "
    print(f"{_iso}  [{_state}] {_what}")
_binding = sum(date.fromisoformat(d) <= AS_OF for d in OMNIBUS)
print(f"\n{_binding} of {len(OMNIBUS)} rungs already past on {AS_OF.isoformat()}: the article "
      "text is settled,\nand the evidence is what has to be built before the dates that are not.")

## 2. The evidence ledger, and why order is evidence

The hardest requirement in this lesson is not statistical. It is that **a floor chosen after
you have seen the results is not a floor**. Everybody agrees with that sentence and almost
nobody's evidence can prove they obeyed it, because the ordering lives in somebody's memory.

So the ordering lives in an append-only ledger instead — the same shape as the P01-L01 log,
minus the parts that lesson grades. Sealing a floor writes an entry. Measuring writes an
entry. Sequence numbers are the proof, and in section 6 you will implement the refusal.

In [ ]:
GENESIS = "0" * 64


class EvidenceLedger:
    """Append-only, sequence-numbered, hash-chained. Given to you; not graded.

    Example:
        >>> led = EvidenceLedger()
        >>> e = led.append("floor", {"metric": "accuracy", "floor": 0.86})
        >>> e["seq"], e["prev"] == GENESIS
        (0, True)
    """

    def __init__(self, system_id: str = SYSTEM_ID) -> None:
        self.system_id = system_id
        self._entries: list[dict] = []

    def append(self, kind: str, payload: dict) -> dict:
        prev = self._entries[-1]["digest"] if self._entries else GENESIS
        entry = {"seq": len(self._entries), "kind": str(kind),
                 "payload": dict(payload), "prev": prev}
        entry["digest"] = digest_of(entry)
        self._entries.append(entry)
        return dict(entry)

    @property
    def entries(self) -> tuple:
        return tuple(dict(e) for e in self._entries)

    def find(self, kind: str | None = None, metric: str | None = None) -> tuple:
        """Every entry matching `kind` and/or the `metric` inside its payload, in order."""
        return tuple(dict(e) for e in self._entries
                     if (kind is None or e["kind"] == kind)
                     and (metric is None or e["payload"].get("metric") == metric))


_demo = EvidenceLedger()
_demo.append("floor", {"metric": "accuracy", "floor": 0.86})
_demo.append("measurement", {"metric": "accuracy", "label": "baseline"})
print(f"{'seq':>3s}  {'kind':12s} {'digest':10s} payload")
for _e in _demo.entries:
    print(f"{_e['seq']:>3d}  {_e['kind']:12s} {_e['digest'][:8]}…  {_e['payload']}")
print(f"\nfloor entries: {len(_demo.find(kind='floor'))} · "
      f"measurements of accuracy: {len(_demo.find('measurement', 'accuracy'))} · "
      f"and seq {_demo.find(kind='floor')[0]['seq']} < "
      f"{_demo.find(kind='measurement')[0]['seq']} is the whole argument")

## 3. The system, its evaluation set and the metrics

Nothing is loaded from disk and nothing is downloaded. The applicants below are **synthetic**,
generated in this cell from one seed, and the deployed scorer is a fixed linear model over
four standardised features. Two runs of this notebook agree to the last digit.

The model is deliberately small. That matters in section 11: a four-feature linear scorer is
far cheaper to steal than a real one, and the lesson says so out loud rather than pretending
the measured number generalises.

In [ ]:
DATA_SEED = 20281115
N_TRAIN, N_TEST = 2000, 1200

FEATURES = ("income", "dti", "months_employed", "prior_defaults")
# The standardisation constants are BAKED INTO the deployed model, as they are in a real
# serving pipeline. That is what makes the unit-change perturbation in section 8 bite.
FEATURE_MEANS = {"income": 52000.0, "dti": 0.34, "months_employed": 62.0, "prior_defaults": 0.35}
FEATURE_SDS = {"income": 21000.0, "dti": 0.14, "months_employed": 46.0, "prior_defaults": 0.70}
MODEL_W = np.array([1.05, -1.60, 0.55, -1.15])
MODEL_B = 0.15
_TRUE_W = np.array([1.00, -1.75, 0.60, -1.30]) * 2.4     # the world, which the model approximates
_TRUE_B = 0.10 * 2.4
DECISION_THRESHOLD = 0.5
IMPUTE_DTI = 0.33            # what the serving pipeline substitutes for a missing ratio


def standardise(features: dict) -> np.ndarray:
    """The four features as a (n, 4) standardised matrix. Given to you; not graded."""
    return np.column_stack([(np.asarray(features[k], dtype=float) - FEATURE_MEANS[k])
                            / FEATURE_SDS[k] for k in FEATURES])


def score(features: dict) -> np.ndarray:
    """The deployed model's score in [0, 1]. Given to you; not graded."""
    z = standardise(features) @ MODEL_W + MODEL_B
    return 1.0 / (1.0 + np.exp(-z))


def predict(features: dict) -> np.ndarray:
    """The deployed model's decision, 1 = approve. Given to you; not graded."""
    return (score(features) >= DECISION_THRESHOLD).astype(int)


def _draw(n: int, rng: np.random.Generator) -> tuple:
    income = np.clip(np.round(rng.lognormal(math.log(48000), 0.42, n)), 9000, 260000)
    dti = np.clip(rng.normal(0.34, 0.14, n), 0.02, 0.92)
    months = np.clip(np.round(rng.gamma(2.2, 28.0, n)), 0, 420)
    prior = rng.poisson(0.35, n).astype(float)
    feats = {"income": income, "dti": dti, "months_employed": months, "prior_defaults": prior}
    p = 1.0 / (1.0 + np.exp(-(standardise(feats) @ _TRUE_W + _TRUE_B)))
    return feats, (rng.random(n) < p).astype(int)


_rng = np.random.default_rng(DATA_SEED)
TRAIN_FEATURES, TRAIN_LABELS = _draw(N_TRAIN, _rng)
TEST_FEATURES, TEST_LABELS = _draw(N_TEST, _rng)
print(f"train {N_TRAIN} rows · test {N_TEST} rows · features {', '.join(FEATURES)}")
print(f"test positives {TEST_LABELS.mean():.4f} · deployed approvals "
      f"{predict(TEST_FEATURES).mean():.4f}")

### The metrics, written so they reduce over the last axis

Every metric below takes `(y_true, y_pred)` and reduces over the **last** axis. That one
convention is what lets the bootstrap you are about to write handle two thousand resamples in
a single array operation instead of a two-thousand-iteration Python loop.

In [ ]:
def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Share of rows the decision got right. Given to you; not graded."""
    return (np.asarray(y_true) == np.asarray(y_pred)).mean(axis=-1)


def recall(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Share of true positives the decision approved. Given to you; not graded."""
    t, p = np.asarray(y_true), np.asarray(y_pred)
    pos = (t == 1).sum(axis=-1)
    return np.where(pos == 0, np.nan, ((t == 1) & (p == 1)).sum(axis=-1) / np.maximum(pos, 1))


def false_positive_rate(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Share of true negatives the decision approved anyway. Given to you; not graded."""
    t, p = np.asarray(y_true), np.asarray(y_pred)
    neg = (t == 0).sum(axis=-1)
    return np.where(neg == 0, np.nan, ((t == 0) & (p == 1)).sum(axis=-1) / np.maximum(neg, 1))


def rate(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Mean of `y_true` alone, for a flag rate. Ignores y_pred. Given to you; not graded.

    `mean` already returns a float; casting a resampled index matrix to float first would
    double this function's peak memory for no change in the answer.
    """
    return np.asarray(y_true).mean(axis=-1)


METRICS = {"accuracy": accuracy, "recall": recall,
           "false_positive_rate": false_positive_rate, "rate": rate}
CI_LEVEL = 0.95
N_BOOT = 2000               # headline metrics
N_BOOT_SWEEP = 500          # 24 sweep points; the shape of the curve, not a headline
N_BOOT_POISON = 1500
N_BOOT_EXTRACT = 1500
BOOT_SEED = 90210           # fixed so the evidence re-runs; it does not make the interval narrower
print(f"{len(METRICS)} metrics · level {CI_LEVEL} · B {N_BOOT} headline / {N_BOOT_SWEEP} sweep "
      f"· seed {BOOT_SEED}")

## 4. Exercise 1 — the paired bootstrap

The percentile bootstrap: resample the evaluation set with replacement, recompute the metric,
and read the 2.5th and 97.5th percentiles of the resulting spread. One decision makes or
breaks it. You must resample **rows**, carrying each row's label and its prediction together.
Draw two independent index vectors — one for the labels, one for the predictions — and you
have measured the accuracy of a model answering somebody else's questions.

Vectorise it. Draw an `(n_boot, n)` matrix of indices in one call, index both arrays with it,
and let the metric reduce over the last axis. The demo after the checks shows what the unpaired
version reports on a model that is right about every single row.

<details><summary>💡 Hint 1 — what to think about</summary>

A resample is a draw of applicants, and each applicant brings a label AND a prediction with
them. What happens to a model that is right about every row if the labels and predictions
are drawn with different indices? Then three smaller traps the check names: is `point` the
metric on the full sample or the average of the replicates; are the percentiles placed from
`level` or typed in; and what can honestly be said when there are no rows at all?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Deal with the degenerate case first: no rows or no resamples gets the blind answer the
docstring gives. Otherwise build the generator from `seed`, make ONE draw of integer row
indices with replacement, shaped resamples by rows, and index both arrays with that same
matrix — when `y_pred` is None, the labels stand in for both. The metric reduces the last
axis to one value per resample. `point` comes from the untouched arrays, the bounds are
percentiles of the replicates placed from `level`, and the result echoes the level, the row
count, the resample count and the method.
</details>

In [ ]:
def bootstrap_ci(y_true: np.ndarray, y_pred: np.ndarray | None = None,
                 metric: Callable = accuracy, n_boot: int = N_BOOT,
                 level: float = CI_LEVEL, seed: int = BOOT_SEED) -> dict:
    """A metric and its percentile bootstrap interval, computed by resampling ROWS.

    Draw `n_boot` resamples of the `n` row indices WITH replacement, in one `(n_boot, n)`
    array, index `y_true` and `y_pred` with the SAME indices, and let `metric` reduce over the
    last axis. `point` is the metric on the full sample, not the mean of the replicates.
    `lo` and `hi` are the `(1-level)/2` and `1-(1-level)/2` percentiles of the replicates.

    Pass `y_pred=None` for a metric that is a property of one vector (`rate`); y_pred is then
    y_true. Use `np.random.default_rng(seed)`, so two runs of the pack agree.

    With no rows, or no resamples, return the blind answer: `point` is nan and the interval is
    the whole range [0.0, 1.0]. Returning a zero-width interval there would claim certainty
    about a measurement nobody made.

    Returns a dict with keys point, lo, hi, level, n, n_boot, method.

    Example:
        >>> t = np.array([1, 0, 1, 0, 1, 0, 1, 0])
        >>> ci = bootstrap_ci(t, t, n_boot=200)
        >>> ci["point"], ci["lo"], ci["hi"]     # every row right, every resample right
        (1.0, 1.0, 1.0)
        >>> ci["n"], ci["method"]
        (8, 'percentile bootstrap')
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_bootstrap() -> None:
    t = np.tile([1, 0], 24)
    perfect = bootstrap_ci(t, t, n_boot=300)
    assert perfect["point"] == 1.0, (
        f"a model right about every row scores 1.0, not {perfect['point']!r}"
    )
    assert perfect["lo"] == 1.0 and perfect["hi"] == 1.0, (
        f"every resample of a perfect model is still perfect, so the interval is [1.0, 1.0]; "
        f"you got [{perfect['lo']:.4f}, {perfect['hi']:.4f}]. An interval below 1 here means "
        "you drew SEPARATE indices for y_true and y_pred and broke the pairing"
    )
    rng = np.random.default_rng(4)
    truth = (rng.random(400) < 0.5).astype(int)
    guess = np.where(rng.random(400) < 0.25, 1 - truth, truth)   # right about 3 rows in 4
    ci = bootstrap_ci(truth, guess, n_boot=800)
    assert ci["point"] == float(accuracy(truth, guess)), (
        "point must be the metric on the FULL sample; the mean of the replicates is a "
        "different number and it is not the one you declare"
    )
    assert ci["lo"] < ci["point"] < ci["hi"], (
        f"the point must sit inside its own interval; you got lo={ci['lo']:.4f} "
        f"point={ci['point']:.4f} hi={ci['hi']:.4f}"
    )
    assert 0.02 < ci["hi"] - ci["lo"] < 0.12, (
        f"a 95% interval on 400 rows at 75% accuracy is about 0.085 wide; yours is "
        f"{ci['hi'] - ci['lo']:.4f}. Far narrower usually means you resampled WITHOUT "
        "replacement (a permutation of the rows is the same rows); far wider usually means "
        "the percentiles are not at 2.5 and 97.5"
    )
    narrow = bootstrap_ci(truth, guess, n_boot=800, level=0.50)
    assert narrow["hi"] - narrow["lo"] < ci["hi"] - ci["lo"], (
        "a 50% interval must be narrower than a 95% one — use the `level` argument to place "
        "the percentiles rather than hard-coding 2.5 and 97.5"
    )
    assert narrow["level"] == 0.50 and ci["n"] == 400 and ci["n_boot"] == 800, (
        "report back the level, the row count and the number of resamples: an interval "
        "without them cannot be reproduced or compared"
    )
    again = bootstrap_ci(truth, guess, n_boot=800)
    assert (again["lo"], again["hi"]) == (ci["lo"], ci["hi"]), (
        "two calls with the same seed must agree exactly — build the generator with "
        "np.random.default_rng(seed), not from global numpy state"
    )
    r_ci = bootstrap_ci(truth, guess, metric=recall, n_boot=400)
    assert r_ci["point"] == float(recall(truth, guess)), (
        "the `metric` argument must be used; you appear to have hard-coded accuracy"
    )
    flags = np.array([1, 0, 0, 1, 0, 0, 0, 0, 1, 0])
    one_vector = bootstrap_ci(flags, None, metric=METRICS["rate"], n_boot=400)
    assert one_vector["point"] == 0.3, (
        "y_pred=None means the metric is a property of y_true alone; set y_pred = y_true "
        f"rather than raising. You got {one_vector['point']!r}"
    )
    empty = bootstrap_ci(np.array([]), np.array([]), n_boot=100)
    assert math.isnan(empty["point"]) and (empty["lo"], empty["hi"]) == (0.0, 1.0), (
        "no rows means no measurement: point is nan and the interval is the whole range "
        f"[0.0, 1.0]. You returned point={empty['point']!r} [{empty['lo']}, {empty['hi']}]"
    )
    print(f"exercise 1 looks right — a three-in-four model measured at {ci['point']:.4f} "
          f"[{ci['lo']:.4f}, {ci['hi']:.4f}] on {ci['n']} rows")

In [ ]:
_try("exercise 1", _check_bootstrap)

Run the next cell once your stub works. It measures what the unpaired bootstrap reports about
a model that is right about every single row — the figure is computed here, not quoted.

In [ ]:
def _show_pairing() -> None:
    t = np.tile([1, 0], 300)
    paired = bootstrap_ci(t, t, n_boot=1000)
    rng = np.random.default_rng(1)
    a, b = rng.integers(0, len(t), size=(1000, len(t))), rng.integers(0, len(t), size=(1000, len(t)))
    unpaired = np.asarray(accuracy(t[a], t[b]), dtype=float)
    lo, hi = np.percentile(unpaired, [2.5, 97.5])
    print(f"a model right about all {len(t)} rows")
    print(f"  paired bootstrap   {paired['point']:.4f}  [{paired['lo']:.4f}, {paired['hi']:.4f}]")
    print(f"  unpaired bootstrap {paired['point']:.4f}  [{lo:.4f}, {hi:.4f}]")
    print(f"The unpaired interval is {(hi - lo) / max(paired['hi'] - paired['lo'], 1e-12):.0f}x "
          f"wider and centred near {unpaired.mean():.2f}, because it has measured the model")
    print("against labels belonging to other applicants. It is not a conservative interval.")
    print("It is an interval for a different question.")


_try("pairing demo", _show_pairing, needs=("exercise 1",))

## 5. Exercise 2 — declaring a metric, and refusing to declare a point

Article 15(3) requires the levels of accuracy and the relevant metrics to be declared in the
instructions for use. Two decisions follow, and the second is the one people get wrong.

First, a declaration that is a bare number is not declarable. Your `declare()` refuses it.
Second, **what you declare is the bound, not the point.** For accuracy, higher is better, so
the number you can defend against a challenge is the *lower* confidence bound. For a false
positive rate, lower is better, so the defensible number is the *upper* bound. Getting that
backwards publishes the most flattering end of your own uncertainty.

<details><summary>💡 Hint 1 — what to think about</summary>

Which end of your own uncertainty could you defend against a challenge? Where higher is
better it is one end, where lower is better it is the other, and taking the same end for
both publishes the flattering one twice. Then: how many different ways can something look
like an interval without being one? The check starts with a bare float and gets subtler.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Refuse before you build. Anything that is not a mapping holding every key in
`DECLARATION_KEYS`, a point or bound that is None or nan, an upper bound below the lower, or
a point outside its own bounds raises `UndeclarableMetricError`. Then build a NEW dict —
never write into the one you were handed. The declared value is the lower bound when higher
is better and the upper bound otherwise; the width is upper minus lower; and the name,
direction, level, method, row count and resample count travel with it.
</details>

In [ ]:
DECLARATION_KEYS = ("point", "lo", "hi", "level", "n", "n_boot", "method")


def declare(name: str, ci: dict, higher_is_better: bool = True) -> dict:
    """Turn a measured interval into one line of the instructions for use.

    Refuse anything that is not a usable interval by raising `UndeclarableMetricError`: a
    mapping missing any of `DECLARATION_KEYS`; a `point`, `lo` or `hi` that is None or nan;
    `hi` below `lo`; or a `point` outside its own `[lo, hi]`.

    `declared` is the end of the interval you can defend: `lo` when higher is better, `hi`
    when lower is better. `width` is `hi - lo`.

    Returns a dict with keys name, point, lo, hi, level, width, declared, direction, method,
    n, n_boot.

    Example:
        >>> ci = {"point": 0.89, "lo": 0.87, "hi": 0.91, "level": 0.95,
        ...       "n": 1200, "n_boot": 2000, "method": "percentile bootstrap"}
        >>> d = declare("accuracy", ci)
        >>> d["declared"], d["direction"], round(d["width"], 4)
        (0.87, 'higher_is_better', 0.04)
        >>> declare("false_positive_rate", ci, higher_is_better=False)["declared"]
        0.91
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_declare() -> None:
    good = {"point": 0.89, "lo": 0.87, "hi": 0.91, "level": 0.95,
            "n": 1200, "n_boot": 2000, "method": "percentile bootstrap"}
    up = declare("accuracy", good)
    assert up["declared"] == 0.87, (
        f"accuracy is higher-is-better, so the number you can defend is the LOWER bound 0.87; "
        f"you declared {up['declared']}"
    )
    assert up["direction"] == "higher_is_better" and abs(up["width"] - 0.04) < 1e-12, (
        "carry the direction and the width: a declaration a reader cannot tell the direction "
        "of is a declaration they cannot check"
    )
    down = declare("false_positive_rate", good, higher_is_better=False)
    assert down["declared"] == 0.91, (
        f"for a rate where lower is better the defensible end is the UPPER bound 0.91, not "
        f"{down['declared']} — declaring the lower end publishes the flattering end of your "
        "own uncertainty"
    )
    assert up["n"] == 1200 and up["n_boot"] == 2000 and up["method"] == good["method"], (
        "the sample size, the resample count and the method travel with the declaration"
    )
    for label, bad in (
            ("a bare float", 0.89),
            ("no interval", {"point": 0.89, "level": 0.95, "n": 5, "n_boot": 5, "method": "x"}),
            ("no method", {"point": 0.89, "lo": 0.87, "hi": 0.91, "level": 0.95,
                           "n": 5, "n_boot": 5}),
            ("nan bound", {**good, "lo": float("nan")}),
            ("none bound", {**good, "hi": None}),
            ("inverted", {**good, "lo": 0.95, "hi": 0.80}),
            ("point outside", {**good, "point": 0.99})):
        try:
            declare("accuracy", bad)
        except UndeclarableMetricError:
            continue
        raise AssertionError(
            f"declare() accepted {label}: {bad!r}. Every one of these is a metric somebody "
            "could publish and nobody could check, which is what the refusal exists to stop"
        )
    print(f"exercise 2 looks right — accuracy declares {up['declared']:.4f} (not the "
          f"{up['point']:.4f} point), fpr declares {down['declared']:.4f}")

In [ ]:
_try("exercise 2", _check_declare)

## 6. Exercise 3 — sealing the floor before you look

A declared floor is the promise the robustness evidence is measured against: *this system
will not go below here*. It is worth exactly as much as the ordering behind it. Seal 0.86
before the sweep and a crossing at severity 0.10 is a finding; pick 0.80 after the sweep and
the same curve is a clean bill of health.

So `seal_floor()` refuses. It refuses to seal a floor for a metric the ledger has already
measured, and it refuses to seal a **second** floor for a metric that already has one —
because re-sealing after the fact is the same cheat wearing a hat. It also refuses a floor
with no rationale, since a bare number is the thing that later gets described as principled.

<details><summary>💡 Hint 1 — what to think about</summary>

There are two ways to seal a floor too late — after the metric has been measured, and after
a floor for it already exists — and both are one question put to the ledger. Put it about
THIS metric: a measurement of some other metric must not block you. And what exactly does
the digest cover, so that somebody downstream can recompute it from the sealed fields alone?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Ask the ledger's `find` for earlier entries of each kind carrying this metric, and raise
`FloorAfterTheFactError` if either exists. Raise `ValueError` for a floor outside [0, 1]
(both ends allowed) or a rationale that is not a string, or is shorter than
`MIN_RATIONALE_CHARS` once stripped. Build the four-field payload with the date as ISO text,
append it as a floor entry, and return the payload plus that entry's `seq` and `digest_of`
the payload — not of the entry, which also carries the seq and the chain.
</details>

In [ ]:
MIN_RATIONALE_CHARS = 20
ACCURACY_FLOOR = 0.86
ACCURACY_FLOOR_RATIONALE = (
    "the manual adjudication queue this system replaces was measured at 0.86 accuracy on the "
    "2025 audit sample; the system may not perform worse than the process it replaces"
)


def seal_floor(ledger: EvidenceLedger, metric: str, floor: float, rationale: str,
               as_of: date = AS_OF) -> dict:
    """Commit a performance floor to the ledger, before anything is measured against it.

    Raise `FloorAfterTheFactError` when the ledger already holds a `"measurement"` entry for
    this metric, and when it already holds a `"floor"` entry for it. Raise `ValueError` when
    the floor is outside [0, 1] or the rationale is shorter than `MIN_RATIONALE_CHARS` after
    stripping.

    Append one `"floor"` entry whose payload is exactly
    `{"metric", "floor", "rationale", "as_of"}`, and return that payload plus the entry's
    `seq` and a `digest` over the payload — so a floor edited afterwards no longer matches
    its own digest.

    Returns a dict with keys seq, metric, floor, rationale, as_of, digest.

    Example:
        >>> led = EvidenceLedger()
        >>> sealed = seal_floor(led, "accuracy", 0.86, "x" * 25)
        >>> sealed["seq"], sealed["floor"], sealed["digest"] == digest_of(
        ...     {"metric": "accuracy", "floor": 0.86, "rationale": "x" * 25,
        ...      "as_of": AS_OF.isoformat()})
        (0, 0.86, True)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_seal_floor() -> None:
    led = EvidenceLedger()
    sealed = seal_floor(led, "accuracy", ACCURACY_FLOOR, ACCURACY_FLOOR_RATIONALE)
    assert sealed["seq"] == 0 and sealed["floor"] == ACCURACY_FLOOR, (
        f"the first seal goes in at seq 0 carrying the floor itself; you returned {sealed!r}"
    )
    assert sealed["digest"] == digest_of({"metric": "accuracy", "floor": ACCURACY_FLOOR,
                                          "rationale": ACCURACY_FLOOR_RATIONALE,
                                          "as_of": AS_OF.isoformat()}), (
        "the digest must be digest_of the payload dict exactly — metric, floor, rationale, "
        "as_of and nothing else — or nobody downstream can recompute it"
    )
    assert len(led.find(kind="floor")) == 1, "the seal must actually append to the ledger"
    try:
        seal_floor(led, "accuracy", 0.80, ACCURACY_FLOOR_RATIONALE)
    except FloorAfterTheFactError:
        pass
    else:
        raise AssertionError(
            "a SECOND floor for the same metric was accepted. Seal 0.86, look at the sweep, "
            "seal 0.80: the ordering check is defeated one entry later unless you refuse this"
        )
    after = EvidenceLedger()
    after.append("measurement", {"metric": "accuracy", "label": "baseline", "point": 0.89})
    try:
        seal_floor(after, "accuracy", 0.80, ACCURACY_FLOOR_RATIONALE)
    except FloorAfterTheFactError:
        pass
    else:
        raise AssertionError(
            "a floor was sealed for a metric the ledger had ALREADY measured. That is the one "
            "thing this function exists to refuse"
        )
    other = EvidenceLedger()
    other.append("measurement", {"metric": "recall", "label": "baseline", "point": 0.90})
    other_sealed = seal_floor(other, "accuracy", 0.86, ACCURACY_FLOOR_RATIONALE)
    assert other_sealed["seq"] == 1, (
        "a measurement of a DIFFERENT metric must not block this floor — match on the metric "
        "in the payload, not merely on the entry kind"
    )
    for label, bad in (("empty", ""), ("short", "too short"), ("not a string", None)):
        try:
            seal_floor(EvidenceLedger(), "accuracy", 0.86, bad)
        except ValueError:
            continue
        raise AssertionError(f"a floor with a {label} rationale was accepted")
    try:
        seal_floor(EvidenceLedger(), "accuracy", 1.4, ACCURACY_FLOOR_RATIONALE)
    except ValueError:
        pass
    else:
        raise AssertionError("a floor of 1.4 was accepted; an accuracy floor lives in [0, 1]")
    print(f"exercise 3 looks right — floor {sealed['floor']} sealed at seq {sealed['seq']}, "
          f"digest {sealed['digest'][:8]}…")

In [ ]:
_try("exercise 3", _check_seal_floor)

The cell below is the demonstration. It runs the honest order, then runs the dishonest one and
prints what the ledger says about it. Nothing here is an instruction you are asked to obey;
it is a refusal you can watch happen.

In [ ]:
def _show_ordering() -> None:
    honest = EvidenceLedger()
    sealed = seal_floor(honest, "accuracy", ACCURACY_FLOOR, ACCURACY_FLOOR_RATIONALE)
    honest.append("measurement", {"metric": "accuracy", "label": "baseline", "point": 0.8900})
    print(f"honest order    : floor at seq {sealed['seq']}, first measurement at seq "
          f"{honest.find(kind='measurement')[0]['seq']} — the floor is prior, so the crossing "
          "report will run")
    crooked = EvidenceLedger()
    crooked.append("measurement", {"metric": "accuracy", "label": "baseline", "point": 0.8900})
    crooked.append("measurement", {"metric": "accuracy", "label": "worst_case", "point": 0.7550})
    try:
        seal_floor(crooked, "accuracy", 0.75, "chosen to reflect observed operating conditions")
    except FloorAfterTheFactError as exc:
        print(f"after the fact  : {exc}")
    print("\nThe second floor is the one a pack would actually contain, and the sentence that")
    print("would accompany it is the one in the except branch above. The ledger is the only")
    print("part of this that a reviewer can check without taking somebody's word.")


_try("ordering demo", _show_ordering, needs=("exercise 3",))

## 7. The headline metrics, measured and declared

`measure()` is given: it computes a metric with your interval **and** writes it to the ledger,
so the ordering evidence accumulates without anybody remembering to record it. Notice the
order this cell runs in — the floor first, then everything else.

In [ ]:
def measure(ledger: EvidenceLedger, metric_name: str, label: str,
            y_true: np.ndarray, y_pred: np.ndarray | None = None,
            n_boot: int = N_BOOT, seed: int = BOOT_SEED) -> dict:
    """Measure a metric with its interval and record it. Given to you; not graded.

    Returns your `bootstrap_ci` result plus `metric`, `label` and the ledger `seq` it landed
    at, so a curve built from these can prove when each point was taken.
    """
    ci = bootstrap_ci(y_true, y_pred, metric=METRICS[metric_name], n_boot=n_boot, seed=seed)
    entry = ledger.append("measurement", {"metric": metric_name, "label": label,
                                          "point": ci["point"], "lo": ci["lo"], "hi": ci["hi"],
                                          "n": ci["n"], "n_boot": ci["n_boot"]})
    return {**ci, "metric": metric_name, "label": label, "seq": entry["seq"]}


def _show_headline() -> None:
    led = EvidenceLedger()
    floor = seal_floor(led, "accuracy", ACCURACY_FLOOR, ACCURACY_FLOOR_RATIONALE)
    y_pred = predict(TEST_FEATURES)
    print(f"floor sealed at seq {floor['seq']} · now measuring\n")
    print(f"{'metric':22s} {'point':>8s} {'95% CI':>20s} {'declared':>9s}  direction")
    for name, higher in (("accuracy", True), ("recall", True), ("false_positive_rate", False)):
        ci = measure(led, name, "baseline", TEST_LABELS, y_pred)
        d = declare(name, ci, higher_is_better=higher)
        print(f"{d['name']:22s} {d['point']:>8.4f} "
              f"{'[' + format(d['lo'], '.4f') + ', ' + format(d['hi'], '.4f') + ']':>20s} "
              f"{d['declared']:>9.4f}  {d['direction']}")
    acc = [e for e in led.find("measurement", "accuracy")][0]
    print(f"\nThe sentence for the instructions for use is not 'accuracy {acc['payload']['point']:.2f}'.")
    print(f"It is: accuracy at least {acc['payload']['lo']:.4f} on {acc['payload']['n']} held-out "
          f"applications,\n95% percentile bootstrap, B = {acc['payload']['n_boot']} — against a "
          f"floor of {floor['floor']} sealed on {floor['as_of']}.")


_try("headline metrics", _show_headline, needs=("exercise 1", "exercise 2", "exercise 3"))

## 8. Exercise 4 — the perturbation sweep

Robustness under Article 15(4) is resilience to errors, faults and inconsistencies in the
system *or its environment*. Four families, each a real integration failure rather than an
adversarial construction:

- **missing_field** — the debt-to-income ratio is absent and the pipeline substitutes its
  documented default.
- **unit_change** — an upstream feed starts reporting income in thousands.
- **sensor_noise** — plausible measurement error on income, as a multiple of its own spread.
- **stale_feature** — the prior-defaults counter is a cycle behind and misses the newest one.

Their severities are **not comparable across families**: one is a share of rows, another a
multiple of a standard deviation. Each family carries its own unit, and the report prints it.

<details><summary>💡 Hint 1 — what to think about</summary>

Every point on this curve has to be provable later, which means it has to land in the
ledger with its own sequence number: which of the two measuring functions you have does
that, and which does not? And each severity must be measured on a freshly damaged copy of
the UNTOUCHED inputs. What happens to a later severity if an earlier one's damage is still
sitting in the arrays?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Look the family up in `PERTURBATIONS`. For each severity, in the order given, hand the
original features to the family's `apply`, predict on what comes back, and record it through
`measure()` with the family-at-severity label, the unchanged labels and the `n_boot`
argument. Add the severity to each result. Return the points with the family and the metric,
and with the feature, unit and expected operating severity copied from the table.
</details>

In [ ]:
def _subset(n: int, fraction: float, seed: int) -> np.ndarray:
    """A deterministic subset of `fraction` of the row indices. Given to you; not graded."""
    return np.random.default_rng(seed).permutation(n)[:int(round(float(fraction) * n))]


def _missing_field(features: dict, severity: float) -> dict:
    out = {k: np.array(v, dtype=float) for k, v in features.items()}
    out["dti"][_subset(len(out["dti"]), severity, 101)] = IMPUTE_DTI
    return out


def _unit_change(features: dict, severity: float) -> dict:
    out = {k: np.array(v, dtype=float) for k, v in features.items()}
    rows = _subset(len(out["income"]), severity, 202)
    out["income"][rows] = out["income"][rows] / 1000.0
    return out


def _sensor_noise(features: dict, severity: float) -> dict:
    out = {k: np.array(v, dtype=float) for k, v in features.items()}
    noise = np.random.default_rng(303).normal(0.0, float(severity) * FEATURE_SDS["income"],
                                              len(out["income"]))
    out["income"] = np.clip(out["income"] + noise, 1000.0, None)
    return out


def _stale_feature(features: dict, severity: float) -> dict:
    out = {k: np.array(v, dtype=float) for k, v in features.items()}
    rows = _subset(len(out["prior_defaults"]), severity, 404)
    out["prior_defaults"][rows] = np.clip(out["prior_defaults"][rows] - 1.0, 0.0, None)
    return out


PERTURBATIONS = {
    "missing_field": {
        "apply": _missing_field, "feature": "dti",
        "severities": (0.0, 0.05, 0.10, 0.20, 0.40, 0.80),
        "unit": "share of applications with the ratio absent and defaulted",
        "expected_operating_severity": 0.05,
        "why": "5% of applications arrive without a verifiable debt-to-income ratio"},
    "unit_change": {
        "apply": _unit_change, "feature": "income",
        "severities": (0.0, 0.01, 0.02, 0.05, 0.10, 0.20),
        "unit": "share of applications whose income arrives in thousands",
        "expected_operating_severity": 0.02,
        "why": "one of the fifty partner feeds has shipped the wrong unit twice in two years"},
    "sensor_noise": {
        "apply": _sensor_noise, "feature": "income",
        "severities": (0.0, 0.10, 0.25, 0.50, 1.00, 2.00),
        "unit": "noise standard deviation as a multiple of the feature's own",
        "expected_operating_severity": 0.25,
        "why": "self-declared income reconciles to payroll within about a quarter of a sd"},
    "stale_feature": {
        "apply": _stale_feature, "feature": "prior_defaults",
        "severities": (0.0, 0.10, 0.25, 0.50, 0.75, 1.00),
        "unit": "share of applications whose default counter is one cycle behind",
        "expected_operating_severity": 0.50,
        "why": "the bureau refresh is monthly and half the queue is scored mid-cycle"},
}
print(f"{'family':16s} {'feature':16s} {'expected':>9s}  grid")
for _f, _s in PERTURBATIONS.items():
    print(f"{_f:16s} {_s['feature']:16s} {_s['expected_operating_severity']:>9.2f}  "
          f"{', '.join(format(x, 'g') for x in _s['severities'])}")
print(f"\n{len(PERTURBATIONS)} families · "
      f"{sum(len(s['severities']) for s in PERTURBATIONS.values())} measurements to take")

In [ ]:
def perturbation_curve(ledger: EvidenceLedger, family: str, features: dict, y_true: np.ndarray,
                       metric_name: str = "accuracy", n_boot: int = N_BOOT_SWEEP) -> dict:
    """Measure the metric at every severity of one perturbation family.

    For each severity in `PERTURBATIONS[family]["severities"]`, in the order given, apply the
    family's `apply` function to `features`, re-run `predict`, and record the result through
    `measure()` with the label `f"{family}@{severity:g}"`. `y_true` never changes: the
    perturbation damages the INPUTS, not the outcomes those applicants actually had.

    `apply` already returns a fresh dict, so `features` itself must come back untouched.

    Returns a dict with keys family, metric, feature, severity_unit,
    expected_operating_severity and points; each point is the `measure()` result with a
    `severity` key added.

    Example:
        >>> led = EvidenceLedger()
        >>> c = perturbation_curve(led, "missing_field", TEST_FEATURES, TEST_LABELS, n_boot=50)
        >>> c["family"], len(c["points"]), c["points"][0]["severity"]
        ('missing_field', 6, 0.0)
        >>> c["points"][0]["seq"] < c["points"][1]["seq"]      # measured in order
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_curve() -> None:
    before = {k: np.array(v) for k, v in TEST_FEATURES.items()}
    led = EvidenceLedger()
    seal_floor(led, "accuracy", ACCURACY_FLOOR, ACCURACY_FLOOR_RATIONALE)
    curve = perturbation_curve(led, "missing_field", TEST_FEATURES, TEST_LABELS, n_boot=60)
    assert [p["severity"] for p in curve["points"]] == \
        [float(s) for s in PERTURBATIONS["missing_field"]["severities"]], (
        "one point per declared severity, in the declared order — the curve is read left to "
        "right and a reordered grid reads as a different system"
    )
    for key, val in (("family", "missing_field"), ("feature", "dti"),
                     ("severity_unit", PERTURBATIONS["missing_field"]["unit"]),
                     ("expected_operating_severity", 0.05)):
        assert curve[key] == val, (
            f"the curve must carry {key} = {val!r}; severities mean nothing without their unit"
        )
    for k in TEST_FEATURES:
        assert np.array_equal(TEST_FEATURES[k], before[k]), (
            f"TEST_FEATURES[{k!r}] changed. The apply functions return a fresh dict; if you "
            "wrote into the one you were handed, every later severity is measured on damage "
            "left behind by the earlier ones"
        )
    assert abs(curve["points"][0]["point"] - float(accuracy(TEST_LABELS,
                                                            predict(TEST_FEATURES)))) < 1e-12, (
        "severity 0.0 must reproduce the undamaged baseline exactly; it is the control"
    )
    seqs = [p["seq"] for p in curve["points"]]
    assert seqs == sorted(seqs) and len(set(seqs)) == len(seqs), (
        f"each point needs its own ledger seq, ascending: got {seqs}. Calling bootstrap_ci "
        "directly instead of measure() leaves no seq and no ordering evidence at all"
    )
    assert len(led.find(kind="measurement")) == len(seqs), (
        "every measurement must reach the ledger — that is what makes the ordering checkable"
    )
    assert curve["points"][-1]["point"] < curve["points"][0]["point"], (
        "defaulting the ratio on 80% of applications must cost accuracy; if it does not, "
        "you are probably re-predicting the undamaged features"
    )
    print(f"exercise 4 looks right — missing_field runs "
          f"{curve['points'][0]['point']:.4f} -> {curve['points'][-1]['point']:.4f} "
          f"over {len(curve['points'])} severities")

In [ ]:
_try("exercise 4", _check_curve)

## 9. Exercise 5 — where the floor gives way

Now the two halves meet. `crossing_severity()` reports the first severity at which the metric
falls through the declared floor — and it refuses to report anything at all unless the floor
was sealed **before** the first measurement in the curve, and still matches its own digest.

Which crossing? The one for the number you declared. You declared the lower bound, so the
system stops being defensible when the lower bound falls through, not when the point does.
Both are computed, because the gap between them is the size of the claim you would have been
making.

<details><summary>💡 Hint 1 — what to think about</summary>

Refuse before you report. Three things make a crossing meaningless: a floor for a different
metric, a floor whose contents no longer match its own digest, and a floor sealed after ANY
point in the curve — any, so the comparison is with the earliest point, not the latest. Then
two questions about the report itself: you declared the lower bound, so whose crossing is
the finding? And is a crossing that lands exactly on the expected operating severity
headroom, or a failure?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Check the metric, then recompute `digest_of` from the floor's four payload fields and compare
it with its `digest`, then compare its `seq` with the smallest seq among the points — raising
the matching error each time. Walk the points in order for the first severity whose `lo`
falls under the floor and, separately, the first whose `point` does. Headroom is counted in
grid steps with `grid_index`, from the expected severity to the bound's crossing — not as a
difference of severities — and the verdict fails when the crossing is at or below the
expected severity.
</details>

In [ ]:
def grid_index(severities: list, value: float, tol: float = 1e-9) -> int:
    """Index of `value` on the severity grid. Given to you; not graded.

    Raises ValueError when the value is not on the grid, because a report that interpolates
    between two measured severities is reporting a severity nobody measured.
    """
    for i, s in enumerate(severities):
        if abs(float(s) - float(value)) <= tol:
            return i
    raise ValueError(f"{value!r} is not one of the measured severities {severities}")


def crossing_severity(curve: dict, sealed_floor: dict) -> dict:
    """The severity at which the declared floor gives way, and the proof that it was declared.

    Refuse first, report second:

    * `ValueError` when the floor governs a different metric from the curve.
    * `TamperedFloorError` when `digest_of({"metric", "floor", "rationale", "as_of"})` taken
      from `sealed_floor` does not equal its own `digest`.
    * `FloorAfterTheFactError` when the floor's `seq` is not strictly below every point's
      `seq` in the curve.

    Then walk the points in order: `crossing_severity` is the first severity whose `lo` is
    below the floor, `point_crossing_severity` the first whose `point` is, each None if it
    never happens. `steps_of_headroom` is how many grid steps separate the expected operating
    severity from the crossing, or None when there is no crossing. The verdict is `"fail"`
    when the crossing is at or below the severity the system expects to meet in operation.

    Returns a dict with keys family, metric, severity_unit, floor, floor_seq,
    first_measurement_seq, expected_operating_severity, crossing_severity,
    point_crossing_severity, steps_of_headroom, verdict.

    Example:
        >>> led = EvidenceLedger()
        >>> f = seal_floor(led, "accuracy", 0.86, ACCURACY_FLOOR_RATIONALE)
        >>> c = perturbation_curve(led, "stale_feature", TEST_FEATURES, TEST_LABELS, n_boot=200)
        >>> r = crossing_severity(c, f)
        >>> r["floor_seq"] < r["first_measurement_seq"], r["verdict"] in {"pass", "fail"}
        (True, True)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_crossing() -> None:
    led = EvidenceLedger()
    floor = seal_floor(led, "accuracy", ACCURACY_FLOOR, ACCURACY_FLOOR_RATIONALE)
    curve = perturbation_curve(led, "missing_field", TEST_FEATURES, TEST_LABELS,
                               n_boot=N_BOOT_SWEEP)
    rep = crossing_severity(curve, floor)
    assert rep["floor_seq"] < rep["first_measurement_seq"], (
        "report both sequence numbers: they are the evidence a reviewer actually checks"
    )
    assert rep["crossing_severity"] is not None and rep["point_crossing_severity"] is not None, (
        "missing_field crosses 0.86 on both readings inside this grid"
    )
    assert rep["crossing_severity"] < rep["point_crossing_severity"], (
        f"the LOWER BOUND falls through the floor before the point does — you reported "
        f"crossing {rep['crossing_severity']} and point crossing "
        f"{rep['point_crossing_severity']}. Swapping them declares a robustness the interval "
        "does not support"
    )
    assert rep["steps_of_headroom"] == 1 and rep["verdict"] == "pass", (
        f"missing_field crosses one grid step beyond the severity it expects to meet, so it "
        f"passes with one step of headroom; you reported {rep['steps_of_headroom']} and "
        f"{rep['verdict']!r}"
    )
    stale = crossing_severity(
        perturbation_curve(led, "stale_feature", TEST_FEATURES, TEST_LABELS,
                           n_boot=N_BOOT_SWEEP), floor)
    assert stale["steps_of_headroom"] == 0 and stale["verdict"] == "fail", (
        f"stale_feature crosses AT the severity it expects to operate at — zero steps of "
        f"headroom is a fail, not a pass; you reported {stale['steps_of_headroom']} and "
        f"{stale['verdict']!r}"
    )
    late = EvidenceLedger()
    late_curve = perturbation_curve(late, "missing_field", TEST_FEATURES, TEST_LABELS, n_boot=40)
    late_floor = {**floor, "seq": late_curve["points"][2]["seq"]}
    try:
        crossing_severity(late_curve, late_floor)
    except FloorAfterTheFactError:
        pass
    else:
        raise AssertionError(
            "a floor whose seq falls in the MIDDLE of the curve was accepted. Compare against "
            "the minimum seq in the curve, not against the last one"
        )
    tampered = {**floor, "floor": 0.70}
    try:
        crossing_severity(curve, tampered)
    except TamperedFloorError:
        pass
    else:
        raise AssertionError(
            "a floor edited from 0.86 to 0.70 after sealing was accepted. Recompute the digest "
            "from the four payload fields and compare"
        )
    try:
        crossing_severity({**curve, "metric": "recall"}, floor)
    except ValueError:
        pass
    else:
        raise AssertionError("an accuracy floor was applied to a recall curve")
    print(f"exercise 5 looks right — missing_field crosses at {rep['crossing_severity']:g} on "
          f"the bound and {rep['point_crossing_severity']:g} on the point")

In [ ]:
_try("exercise 5", _check_crossing)

In [ ]:
def _show_sweep() -> None:
    led = EvidenceLedger()
    floor = seal_floor(led, "accuracy", ACCURACY_FLOOR, ACCURACY_FLOOR_RATIONALE)
    print(f"floor {floor['floor']} sealed at seq {floor['seq']} · "
          f"'<' marks the first severity whose LOWER BOUND falls through it\n")
    for family in PERTURBATIONS:
        curve = perturbation_curve(led, family, TEST_FEATURES, TEST_LABELS, n_boot=N_BOOT_SWEEP)
        rep = crossing_severity(curve, floor)
        cells = []
        for p in curve["points"]:
            mark = "<" if p["severity"] == rep["crossing_severity"] else " "
            cells.append(f"{p['severity']:g}:{p['point']:.3f}[{p['lo']:.3f}]{mark}")
        print(f"{family:15s} " + " ".join(cells))
        print(f"{'':15s} expects {rep['expected_operating_severity']:g} · crosses "
              f"{rep['crossing_severity']} (bound) / {rep['point_crossing_severity']} (point) · "
              f"{rep['steps_of_headroom']} steps · {rep['verdict'].upper()}")
        print(f"{'':15s} unit: {rep['severity_unit']}")


_try("sweep table", _show_sweep,
     needs=("exercise 1", "exercise 3", "exercise 4", "exercise 5"))

## 10. Exercise 6 — poisoning, and the batch too small to accuse

Article 15(5) names data poisoning first among the AI-specific vulnerabilities. The detector
here is deliberately simple and needs no labels you do not already have: for every training
row, look at its `k` nearest neighbours in standardised feature space and flag it when almost
none of them shares its label. Genuine label noise scatters; a poisoned upload clusters.

The trap is the same one `P01-L04` set for the bias probe, and it is worth meeting twice. The
loudest batch in the table is a 20-row upload with half its labels flipped. Twenty rows cannot
carry an accusation, so the scan names it and makes no claim about its rate either way —
`MIN_SUPPORT` is the same 30-row floor that lesson used.

<details><summary>💡 Hint 1 — what to think about</summary>

Three decisions carry this scan. When is a batch too small to say anything about, and does
that test come before or after any comparison? What is a batch compared WITH — would the
comparison be fair if the batch sat inside its own baseline? And does "worse than the
baseline" mean a higher point estimate, or two intervals that have come apart?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Flag every row whose neighbour agreement is under `NEIGHBOUR_AGREEMENT_FLOOR`, and measure
the overall flag rate with your bootstrap and the `rate` metric. For each batch id in sorted
order, measure its own rows and, separately, every row outside it. Apply the support floor
before anything else, using the `min_support` argument; only an examinable batch can be
poisoned, and only when its lower bound sits above the baseline's upper bound. Half of each
interval's width, added, is the minimum detectable excess. Each batch's entry records its row
count, whether it cleared the floor, both intervals, that excess and its status; rank the
findings worst rate first.
</details>

In [ ]:
K_NEIGHBOURS = 15
NEIGHBOUR_AGREEMENT_FLOOR = 0.30   # flag a row when under 30% of its neighbours share its label
MIN_SUPPORT = 30                   # rows below which no batch claim is made at all, as in P01-L04
BATCH_SIZES = (420, 360, 300, 240, 200, 160, 120, 90, 90, 20)
BATCH_IDS = tuple(f"b{i:02d}" for i in range(len(BATCH_SIZES)))


def _build_training_labels() -> tuple:
    """Labels as recorded, batch ids, and the planted flips. Given to you; not graded."""
    rng = np.random.default_rng(24601)
    batches = np.concatenate([np.full(s, BATCH_IDS[i]) for i, s in enumerate(BATCH_SIZES)])
    labels = TRAIN_LABELS.copy()
    ordinary = rng.random(N_TRAIN) < 0.03           # ordinary label noise, everywhere
    labels[ordinary] = 1 - labels[ordinary]
    for batch_id, share in (("b06", 0.55), ("b08", 0.12), ("b09", 0.50)):
        rows = np.flatnonzero(batches == batch_id)
        chosen = rows[rng.random(len(rows)) < share]
        labels[chosen] = 1 - labels[chosen]
        # b06 is the poisoned upload; b08 is merely a noisy vendor; b09 is twenty rows.
    return labels, batches


RECORDED_LABELS, TRAIN_BATCHES = _build_training_labels()


def neighbour_agreement(features: dict, labels: np.ndarray, k: int = K_NEIGHBOURS) -> np.ndarray:
    """Share of each row's `k` nearest neighbours that carry the same label. Given; not graded."""
    z = standardise(features)
    sq = (z ** 2).sum(axis=1)
    d2 = sq[:, None] + sq[None, :] - 2.0 * (z @ z.T)
    np.fill_diagonal(d2, np.inf)
    nn = np.argpartition(d2, k, axis=1)[:, :k]
    return (np.asarray(labels)[nn] == np.asarray(labels)[:, None]).mean(axis=1)


print(f"{len(BATCH_SIZES)} upload batches, {N_TRAIN} rows · k = {K_NEIGHBOURS} · "
      f"flag under {NEIGHBOUR_AGREEMENT_FLOOR} agreement · min support {MIN_SUPPORT}")

In [ ]:
def poison_scan(features: dict, labels: np.ndarray, batches: np.ndarray,
                k: int = K_NEIGHBOURS, min_support: int = MIN_SUPPORT,
                n_boot: int = N_BOOT_POISON) -> dict:
    """Flag rows whose neighbourhood disagrees with them, then judge the batches.

    Flag a row when `neighbour_agreement(...) < NEIGHBOUR_AGREEMENT_FLOOR`. For each batch id,
    in sorted order, measure its flag rate with `bootstrap_ci(..., metric=METRICS["rate"])`
    and measure a `baseline` the same way over every row NOT in that batch — leave-one-out,
    because a batch cannot be part of the baseline it is being compared against.

    A batch of fewer than `min_support` rows is `"insufficient_support"`, decided BEFORE any
    comparison. Otherwise it is `"poisoned"` when its interval clears the baseline's — its
    `lo` above the baseline's `hi` — and `"ok"` when it does not. `min_detectable_excess` is
    the smallest excess that test could have found: half this batch's interval width plus half
    the baseline's.

    `findings` are the poisoned ids, worst rate first; `insufficient` are the small ones,
    sorted. The verdict is `"fail"` when there is any finding.

    Returns a dict with keys k, min_support, flagged_rows, overall, batches, findings,
    insufficient, verdict.

    Example:
        >>> rep = poison_scan(TRAIN_FEATURES, RECORDED_LABELS, TRAIN_BATCHES, n_boot=200)
        >>> rep["batches"]["b09"]["status"], rep["batches"]["b09"]["n"]
        ('insufficient_support', 20)
        >>> rep["findings"]
        ['b06']
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_poison() -> None:
    rep = poison_scan(TRAIN_FEATURES, RECORDED_LABELS, TRAIN_BATCHES, n_boot=400)
    assert set(rep["batches"]) == set(BATCH_IDS), (
        f"every batch gets a row, including the small ones; you reported "
        f"{sorted(rep['batches'])}"
    )
    tiny = rep["batches"]["b09"]
    assert tiny["n"] == 20 and tiny["status"] == "insufficient_support", (
        "b09 has 20 rows and the loudest flag rate in the table. Twenty rows cannot carry an "
        "accusation: the support floor is applied BEFORE any comparison, exactly as the "
        "subgroup floor was in P01-L04"
    )
    assert "b09" not in rep["findings"], (
        "the 20-row batch must not appear in findings — that is what the floor is for"
    )
    assert "b09" in rep["insufficient"], (
        "it must still be NAMED: 'cannot be examined' is itself a finding about the data"
    )
    assert rep["findings"] == ["b06"], (
        f"exactly one batch clears the baseline's interval; you reported {rep['findings']}"
    )
    worst = rep["batches"]["b06"]
    assert worst["rate"]["lo"] > worst["baseline"]["hi"], (
        "a batch is poisoned when its interval CLEARS the baseline's, not when its point "
        "estimate is merely higher"
    )
    noisy = rep["batches"]["b08"]
    assert noisy["rate"]["point"] > noisy["baseline"]["point"] and noisy["status"] == "ok", (
        f"b08 has a higher flag rate than its baseline and is still not a finding, because 90 "
        f"rows cannot separate it: its min detectable excess is "
        f"{noisy['min_detectable_excess']:.4f}. Comparing point estimates would accuse a "
        "vendor on the strength of noise"
    )
    assert worst["baseline"]["n"] == N_TRAIN - worst["n"], (
        f"the baseline is leave-one-out: {N_TRAIN} - {worst['n']} rows, not all {N_TRAIN}. "
        "Leaving the batch inside its own baseline hides exactly the batch you are hunting"
    )
    assert rep["batches"]["b00"]["min_detectable_excess"] < \
        rep["batches"]["b07"]["min_detectable_excess"], (
        "a bigger batch detects a smaller excess; if yours does not, the detection limit is "
        "not coming from the interval widths"
    )
    assert rep["verdict"] == "fail" and rep["flagged_rows"] > 0, (
        "one finding means the scan fails, and the flagged row count is part of the record"
    )
    clean = poison_scan(TRAIN_FEATURES, TRAIN_LABELS,
                        np.array(["b00"] * 1000 + ["b01"] * 1000), n_boot=400)
    assert clean["verdict"] == "pass" and clean["findings"] == [], (
        "on labels with nothing planted in them the scan must come back clean — a scan that "
        "always fails carries no information at all"
    )
    print(f"exercise 6 looks right — {rep['flagged_rows']} rows flagged, findings "
          f"{rep['findings']}, insufficient {rep['insufficient']}")

In [ ]:
_try("exercise 6", _check_poison)

In [ ]:
def _show_batches() -> None:
    rep = poison_scan(TRAIN_FEATURES, RECORDED_LABELS, TRAIN_BATCHES)
    print(f"overall flag rate {rep['overall']['point']:.4f} "
          f"[{rep['overall']['lo']:.4f}, {rep['overall']['hi']:.4f}]\n")
    print(f"{'batch':6s} {'n':>5s} {'rate':>7s} {'95% CI':>18s} {'baseline hi':>12s} "
          f"{'mde':>7s}  status")
    for batch_id, cell in rep["batches"].items():
        ci = cell["rate"]
        print(f"{batch_id:6s} {cell['n']:>5d} {ci['point']:>7.4f} "
              f"{'[' + format(ci['lo'], '.4f') + ',' + format(ci['hi'], '.4f') + ']':>18s} "
              f"{cell['baseline']['hi']:>12.4f} {cell['min_detectable_excess']:>7.4f}  "
              f"{cell['status']}")
    b09, b06 = rep["batches"]["b09"], rep["batches"]["b06"]
    print(f"\nb09's rate is {b09['rate']['point']:.4f} and b06's is {b06['rate']['point']:.4f}. "
          f"b09 looks worse by\n{b09['rate']['point'] - b06['rate']['point']:+.4f} and is "
          f"reported as unexaminable, because its interval is "
          f"{(b09['rate']['hi'] - b09['rate']['lo']) / (b06['rate']['hi'] - b06['rate']['lo']):.1f}x "
          "wider\nthan b06's. The number to act on is the one whose uncertainty is small "
          "enough to act on.")


_try("batch table", _show_batches, needs=("exercise 1", "exercise 6"))

## 11. Exercise 7 — how many queries buy you the model

Article 15(5) also names confidentiality attacks. Rate limiting is the control every pack
claims; almost none says how many queries the model is worth, which is the only number that
makes a rate limit a control rather than a setting.

So measure it. An attacker sees decisions only — no scores — fits a least-squares surrogate to
`q` of them, and is scored on how often the surrogate agrees with the deployed model over a
fresh population. The curve is given; you turn it into a budget, and the direction of the
bound **flips**. For your own performance you declare the bound you can defend. For somebody
else's attack you plan against the bound that favours them.

<details><summary>💡 Hint 1 — what to think about</summary>

The budget is an interval too, and each end of it comes from a different end of the
agreement interval. As the query count grows, which bound of agreement reaches the target
first — and is that the budget a defender plans against, or the one to feel confident about?
Then: which end should the recommended rate limit be sized from, and is a client running
at exactly the limit over it?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Walk the grid in order for three first arrivals: the first budget whose UPPER bound reaches
the target is the lucky one, the first whose point does is the middle one, and the first
whose LOWER bound does is the confident one. If the grid never gets there, say so, and leave
the budget and the days as None rather than inventing them. Divide each budget by the
enforced daily limit, size the recommendation from the lucky budget over `target_days`
rounded down, and list the clients strictly above the limit, busiest first.
</details>

In [ ]:
EXTRACTION_TARGET = 0.95
EXTRACTION_GRID = (5, 10, 20, 40, 60, 90, 130, 200, 320, 600)
N_ATTACK_REPEATS = 6
N_EVAL_QUERIES = 1200
CURRENT_DAILY_LIMIT = 500          # the rate limit the service enforces today
TARGET_DAYS = 90                   # how long we want an extraction to take
CLIENT_QUERY_RATES = {"broker-alpha": 42.0, "broker-beta": 610.0, "aggregator-gamma": 118.0,
                      "aggregator-delta": 9.5, "partner-epsilon": 1450.0,
                      "partner-zeta": 77.0, "internal-qa": 240.0, "sandbox-eta": 3.0}


def extraction_curve(grid: tuple = EXTRACTION_GRID, repeats: int = N_ATTACK_REPEATS,
                     n_boot: int = N_BOOT_EXTRACT) -> dict:
    """Agreement between a label-only surrogate and the deployed model. Given; not graded."""
    evaluation, _ = _draw(N_EVAL_QUERIES, np.random.default_rng(31415))
    deployed = predict(evaluation)
    design = np.column_stack([standardise(evaluation), np.ones(len(deployed))])
    points = []
    for q in grid:
        matches = []
        for repeat in range(repeats):
            rng = np.random.default_rng(900000 + 1000 * repeat + q)
            queries, _ = _draw(q, rng)
            targets = predict(queries).astype(float) * 2.0 - 1.0
            a = np.column_stack([standardise(queries), np.ones(q)])
            w, *_ = np.linalg.lstsq(a, targets, rcond=None)
            matches.append(((design @ w >= 0).astype(int) == deployed).astype(int))
        pooled = np.concatenate(matches)
        ci = bootstrap_ci(pooled, None, metric=METRICS["rate"], n_boot=n_boot)
        points.append({"queries": int(q), **ci})
    return {"target_agreement": EXTRACTION_TARGET, "repeats": int(repeats),
            "evaluation_rows": int(N_EVAL_QUERIES), "points": points}

In [ ]:
def extraction_risk(curve: dict, target: float = EXTRACTION_TARGET,
                    current_daily_limit: float = CURRENT_DAILY_LIMIT,
                    target_days: int = TARGET_DAYS,
                    client_rates: dict | None = None) -> dict:
    """Turn an agreement curve into a query budget, a time to extraction and a verdict.

    `queries_to_extract` is an interval over the grid, and the direction flips relative to a
    performance claim. `lo` is the smallest `queries` whose `hi` reaches the target — the
    attacker got lucky, and that is the budget you must plan against. `point` is the smallest
    whose `point` reaches it. `hi` is the smallest whose `lo` does — the budget you would be
    confident about. Each is None when the grid never reaches the target, and `reached` is
    then False.

    `days_to_extract` is that same interval divided by `current_daily_limit`, keeping the
    ordering: the soonest extraction uses the smallest budget. `recommended_daily_limit` is
    `floor(queries_to_extract["lo"] / target_days)`. `clients_over_current_limit` are the ids
    in `client_rates` (default `CLIENT_QUERY_RATES`) above the limit in force, busiest first.
    The verdict is `"fail"` when the soonest extraction lands inside `target_days`.

    Returns a dict with keys target_agreement, reached, queries_to_extract, days_to_extract,
    current_daily_limit, target_days, recommended_daily_limit, clients_over_current_limit,
    verdict.

    Example:
        >>> r = extraction_risk(extraction_curve(grid=(5, 600), repeats=2, n_boot=100))
        >>> r["reached"], r["queries_to_extract"]["lo"] <= r["queries_to_extract"]["hi"]
        (True, True)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_extraction() -> None:
    curve = extraction_curve()
    rep = extraction_risk(curve)
    assert rep["reached"] is True, "this grid does reach 0.95 agreement; check the comparison"
    q = rep["queries_to_extract"]
    assert q["lo"] <= q["point"] <= q["hi"], (
        f"lo <= point <= hi must hold by construction, because hi >= point >= lo at every "
        f"grid budget; you reported {q['lo']}, {q['point']}, {q['hi']}"
    )
    assert q["lo"] == 130 and q["point"] == 200 and q["hi"] == 200, (
        f"on this curve the attacker-lucky budget is 130 queries and the confident one is 200; "
        f"you reported lo={q['lo']} point={q['point']} hi={q['hi']}. Using the LOWER bound of "
        "the agreement for the lucky budget inverts the threat"
    )
    assert abs(rep["days_to_extract"]["lo"] - q["lo"] / CURRENT_DAILY_LIMIT) < 1e-12, (
        "the soonest extraction uses the SMALLEST budget; dividing the largest by the limit "
        "reports a comfort the evidence does not support"
    )
    assert rep["recommended_daily_limit"] == math.floor(130 / TARGET_DAYS), (
        f"the recommendation is floor(lo / target_days) = {math.floor(130 / TARGET_DAYS)}; you "
        f"said {rep['recommended_daily_limit']}. Sizing it off the confident budget sets a "
        "limit the lucky attacker walks straight through"
    )
    assert rep["clients_over_current_limit"] == ["partner-epsilon", "broker-beta"], (
        f"two clients exceed {CURRENT_DAILY_LIMIT} queries a day, busiest first; you reported "
        f"{rep['clients_over_current_limit']}"
    )
    assert rep["verdict"] == "fail", (
        "the soonest extraction lands well inside the 90-day target, so this fails"
    )
    calm = extraction_risk(curve, target=0.999)
    assert calm["reached"] is False and calm["queries_to_extract"] is None, (
        "a target this grid never reaches must be reported as not reached, with no budget "
        "invented for it"
    )
    print(f"exercise 7 looks right — {q['lo']} queries (lucky) / {q['hi']} (confident), "
          f"{rep['days_to_extract']['lo']:.2f} days at the current limit")

In [ ]:
_try("exercise 7", _check_extraction)

In [ ]:
def _show_extraction() -> None:
    curve = extraction_curve()
    rep = extraction_risk(curve)
    print(f"{'queries':>8s} {'agreement':>10s} {'95% CI':>18s}")
    for p in curve["points"]:
        mark = " <- lucky" if p["queries"] == rep["queries_to_extract"]["lo"] else ""
        print(f"{p['queries']:>8d} {p['point']:>10.4f} "
              f"{'[' + format(p['lo'], '.4f') + ',' + format(p['hi'], '.4f') + ']':>18s}{mark}")
    q, d = rep["queries_to_extract"], rep["days_to_extract"]
    print(f"\n{q['lo']} queries reach {rep['target_agreement']} agreement on the lucky reading, "
          f"{q['hi']} on the confident one.")
    print(f"At the enforced {rep['current_daily_limit']:.0f} a day that is {d['lo']:.2f} days, "
          f"against a target of {rep['target_days']}.")
    limit = rep["recommended_daily_limit"]
    print(f"To buy {rep['target_days']} days the limit would have to be {limit:.0f} "
          f"{'query' if limit == 1 else 'queries'} a day, which is not a service.")
    print("\nThat is the finding, and it is not 'tighten the rate limit'. A four-feature linear")
    print("scorer is worth a couple of hundred queries and no rate limit reaches that far; the")
    print("control has to be something else. A real model with hundreds of features would move")
    print("this number by orders of magnitude — which is why it is measured per system and not")
    print("quoted from a paper.")


_try("extraction table", _show_extraction, needs=("exercise 1", "exercise 7"))

## 12. Exercise 8 — the residual risk the evidence bounds

The residual-risk statement is where packs stop being evidence. It is written last, in prose,
by whoever is left, and it says what everybody hopes. Yours is generated from the report: each
catalogued risk either resolves to an interval in the evidence or it does not, and the ones
that do not are reported as unsupported rather than quietly dropped.

Three of the seven below can never resolve. They are the honest half of the statement.

<details><summary>💡 Hint 1 — what to think about</summary>

A risk you cannot support is still a statement: the point is that the unbounded ones stay on
the page, in catalogue order. So each entry has three possible outcomes — which one of them
is an error rather than an unsupported statement? Think about what a bare number waiting at
the end of an evidence path is claiming.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For each entry in order: with no evidence path, emit an unsupported statement whose reason
is the entry's `blind_spot`. Otherwise split the path on dots and walk the report one key at
a time, all the way down; a key that is not there makes an unsupported statement whose
reason names the path. If the walk arrives, `is_interval` decides: an interval is a
supported statement with it as the bound, and anything else raises
`UndeclarableMetricError`. Then count the supported and list the unsupported ids.
</details>

In [ ]:
def is_interval(node: Any) -> bool:
    """True when `node` is a dict carrying point, lo and hi. Given to you; not graded."""
    return isinstance(node, dict) and all(key in node for key in ("point", "lo", "hi"))


RESIDUAL_RISKS = (
    {"id": "accuracy_below_declared", "evidence": "accuracy.headline",
     "risk": "the system performs below the accuracy declared in the instructions for use",
     "blind_spot": ""},
    {"id": "degradation_under_tested_perturbation",
     "evidence": "robustness.families.stale_feature.at_expected",
     "risk": "accuracy at the severity the worst tested family expects to meet in operation",
     "blind_spot": ""},
    {"id": "poisoned_upload_batch", "evidence": "cybersecurity.poisoning.overall",
     "risk": "manipulated or erroneous labels remain in the training set at the measured "
             "neighbourhood-disagreement rate",
     "blind_spot": ""},
    {"id": "model_extraction_by_one_client",
     "evidence": "cybersecurity.extraction.queries_to_extract",
     "risk": "a single client reconstructs the decision function from its own query traffic",
     "blind_spot": ""},
    {"id": "degradation_under_untested_perturbation", "evidence": None,
     "risk": "an input perturbation outside the four tested families degrades the system",
     "blind_spot": "four families were tested; the space of input faults is not enumerable, "
                   "so nothing here bounds the fifth"},
    {"id": "poisoning_below_the_detection_floor", "evidence": None,
     "risk": "a batch is poisoned at a rate the neighbourhood scan cannot separate from noise",
     "blind_spot": "the scan reports a minimum detectable excess per batch; below it the scan "
                   "is silent, and silence is not absence"},
    {"id": "extraction_by_colluding_clients", "evidence": None,
     "risk": "several clients pool their query traffic to stay under the per-client limit",
     "blind_spot": "the extraction analysis is per client id and cannot see collusion at all"},
)


def residual_risk(report: dict, catalogue: tuple = RESIDUAL_RISKS) -> dict:
    """Emit one statement per catalogued risk, supported only where the evidence bounds it.

    For each entry, in catalogue order: an `evidence` of None is unsupported, and `because` is
    the entry's `blind_spot`. Otherwise resolve the dotted path through `report` — a missing
    key is also unsupported, with `because` naming the path that was not there. A path that
    resolves to something that is not an interval (`point`, `lo` and `hi` all present) is a
    number being offered as evidence without one: raise `UndeclarableMetricError`.

    Each statement is a dict with keys id, risk, supported, bound and because; `bound` is the
    resolved interval, or None.

    Returns a dict with keys statements, n_supported and unsupported.

    Example:
        >>> ci = {"point": 0.89, "lo": 0.87, "hi": 0.91}
        >>> out = residual_risk({"a": {"b": ci}},
        ...                     ({"id": "x", "evidence": "a.b", "risk": "r", "blind_spot": ""},))
        >>> out["n_supported"], out["statements"][0]["bound"]["lo"]
        (1, 0.87)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_residual() -> None:
    ci = {"point": 0.89, "lo": 0.87, "hi": 0.91, "level": 0.95, "n": 12,
          "n_boot": 12, "method": "percentile bootstrap"}
    report = {"accuracy": {"headline": ci}, "deep": {"nested": {"ci": ci}}}
    catalogue = (
        {"id": "bounded", "evidence": "accuracy.headline", "risk": "r1", "blind_spot": ""},
        {"id": "deep", "evidence": "deep.nested.ci", "risk": "r2", "blind_spot": ""},
        {"id": "absent", "evidence": "accuracy.nowhere", "risk": "r3", "blind_spot": ""},
        {"id": "unbounded", "evidence": None, "risk": "r4", "blind_spot": "nothing bounds it"},
    )
    out = residual_risk(report, catalogue)
    assert [s["id"] for s in out["statements"]] == [c["id"] for c in catalogue], (
        "emit one statement per catalogued risk, in catalogue order — an unsupported risk is "
        "dropped from a written statement and that is what this replaces"
    )
    assert out["n_supported"] == 2 and out["unsupported"] == ["absent", "unbounded"], (
        f"two of these four resolve to an interval; you reported n_supported="
        f"{out['n_supported']} unsupported={out['unsupported']}"
    )
    assert out["statements"][1]["bound"] is ci, (
        "a dotted path has to be walked all the way down, not just one level"
    )
    assert out["statements"][2]["supported"] is False and \
        "accuracy.nowhere" in out["statements"][2]["because"], (
        "a path that is not in the report is unsupported, and the reason must name the path"
    )
    assert out["statements"][3]["because"] == "nothing bounds it", (
        "an entry with no evidence at all carries its catalogue blind_spot as the reason"
    )
    try:
        residual_risk({"accuracy": {"headline": 0.89}}, catalogue[:1])
    except UndeclarableMetricError:
        pass
    else:
        raise AssertionError(
            "a bare 0.89 was accepted as evidence for a residual-risk statement. A number "
            "with no interval cannot bound a risk, and this is the third place the lesson "
            "refuses one"
        )
    print(f"exercise 8 looks right — {out['n_supported']} bounded, "
          f"{len(out['unsupported'])} reported as unbounded rather than dropped")

In [ ]:
_try("exercise 8", _check_residual)

## 13. Exercise 9 — auditing the report for naked numbers

The artefact is only worth something if the rule held everywhere, so audit it. Walk the report
and name every numeric leaf that is neither inside an interval nor one of the quantities that
legitimately has none. There are three such kinds, and the distinction is the whole lesson:
numbers **counted** on the data in front of you, numbers **declared** or chosen by you, and
numbers read off the grid or computed from intervals already reported.

<details><summary>💡 Hint 1 — what to think about</summary>

Three kinds of node want three treatments: an interval, bounded as a whole; a container, to
be walked; and a leaf, to be judged. In Python `True` passes an isinstance test for int — so
which test would report every boolean in the report? And inside a list, which key decides
whether a number may be bare: the index, or the key the list hangs under?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Recurse. A mapping that `is_interval` accepts contributes nothing and is not entered. Any
other mapping is walked with `.key` appended to the path and that key passed down. A list or
tuple is walked with `[i]` appended and the PARENT's key passed down unchanged. A leaf that
is an int or a float but not a bool is reported when its key is not in `EXACT_QUANTITIES`.
Collect everything, and sort once at the end.
</details>

In [ ]:
EXACT_QUANTITIES = frozenset({
    # counted exactly, on data we hold: no sampling, nothing to be uncertain about
    "n", "n_boot", "seq", "floor_seq", "first_measurement_seq", "rows", "flagged_rows",
    "k", "min_support", "n_supported", "repeats", "evaluation_rows",
    # declared or chosen by us: a policy, not an estimate
    "level", "floor", "severity", "expected_operating_severity", "target_agreement",
    "current_daily_limit", "target_days",
    # read off the grid, or derived from the intervals this report already carries
    "crossing_severity", "point_crossing_severity", "steps_of_headroom",
    "min_detectable_excess", "recommended_daily_limit", "queries",
})


def unbounded_numbers(node: Any, path: str = "", key: str = "") -> list:
    """Every numeric leaf in `node` that carries neither an interval nor a licence to be bare.

    Walk it: a dict for which `is_interval` is true is bounded, so do not descend into it; any
    other dict is walked key by key, appending `".key"` to the path; a list or tuple is walked
    by index, appending `"[i]"` and keeping the parent's `key`. An int or float is bounded
    when its `key` is in `EXACT_QUANTITIES` and unbounded otherwise. Booleans are not numbers;
    strings and None are not either.

    Returns a sorted list of the dotted paths of the unbounded numbers.

    Example:
        >>> unbounded_numbers({"accuracy": 0.89, "rows": 1200})
        ['.accuracy']
        >>> unbounded_numbers({"accuracy": {"point": 0.89, "lo": 0.87, "hi": 0.91}})
        []
        >>> unbounded_numbers({"cells": [{"drift": 0.2}, {"n": 5}]})
        ['.cells[0].drift']
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_audit() -> None:
    assert unbounded_numbers({"accuracy": 0.89}) == [".accuracy"], (
        "a bare 0.89 under 'accuracy' is exactly what this audit exists to find; you returned "
        f"{unbounded_numbers({'accuracy': 0.89})}"
    )
    assert unbounded_numbers({"n": 1200, "level": 0.95, "seq": 3}) == [], (
        "a row count, a confidence level and a sequence number are counted or declared, not "
        "estimated — they are in EXACT_QUANTITIES and must not be reported"
    )
    assert unbounded_numbers({"accuracy": {"point": 0.89, "lo": 0.87, "hi": 0.91,
                                           "width": 0.04}}) == [], (
        "do not descend into an interval: everything inside one is bounded by construction, "
        "'width' included"
    )
    assert unbounded_numbers({"cells": [{"drift": 0.2}, {"n": 5}]}) == [".cells[0].drift"], (
        "walk lists by index and keep the parent key for their numeric items"
    )
    assert unbounded_numbers({"ok": True, "name": "x", "missing": None}) == [], (
        "True is not a number here — isinstance(True, int) is True in Python, so a bare "
        "isinstance check on int reports every boolean in the report"
    )
    deep = {"a": {"b": {"c": 0.5, "n": 2}}, "d": [[{"e": 0.1}]]}
    assert unbounded_numbers(deep) == [".a.b.c", ".d[0][0].e"], (
        f"nested dicts and nested lists both have to be walked; you returned "
        f"{unbounded_numbers(deep)}"
    )
    out = unbounded_numbers({"z": 0.1, "a": 0.2})
    assert out == [".a", ".z"], f"return the paths sorted; you returned {out}"
    print("exercise 9 looks right — the audit finds naked numbers and leaves the counted ones")

In [ ]:
_try("exercise 9", _check_audit)

## 14. The artefact

Everything assembles into one Article 15 evidence report. The assembler is given: it seals the
floor first, on a fresh ledger, then measures — so the ordering in the artefact is a property
of the code that built it, not of who ran it.

In [ ]:
_REPORT_CACHE: dict = {}
# The assembler runs exercises 1 to 8 in a row, so every cell that reads the report waits on
# all eight. Exercise 9 audits the finished report; it is not part of building it.
_FOR_REPORT = tuple(f"exercise {k}" for k in range(1, 9))


def article_15_report(as_of: date = AS_OF) -> dict:
    """Assemble the Article 15 evidence report. Given to you — it is your nine pieces in order."""
    if "report" in _REPORT_CACHE:
        return _REPORT_CACHE["report"]
    ledger = EvidenceLedger()
    floor = seal_floor(ledger, "accuracy", ACCURACY_FLOOR, ACCURACY_FLOOR_RATIONALE, as_of)
    y_pred = predict(TEST_FEATURES)
    headline = {
        name: declare(name, measure(ledger, name, "baseline", TEST_LABELS, y_pred), higher)
        for name, higher in (("accuracy", True), ("recall", True),
                             ("false_positive_rate", False))}
    families = {}
    for family in PERTURBATIONS:
        curve = perturbation_curve(ledger, family, TEST_FEATURES, TEST_LABELS,
                                   n_boot=N_BOOT_SWEEP)
        at = grid_index([p["severity"] for p in curve["points"]],
                        curve["expected_operating_severity"])
        families[family] = {"curve": curve, "floor": crossing_severity(curve, floor),
                            "at_expected": curve["points"][at]}
    poison = poison_scan(TRAIN_FEATURES, RECORDED_LABELS, TRAIN_BATCHES)
    extraction = extraction_risk(extraction_curve())
    robust_verdict = ("fail" if any(f["floor"]["verdict"] == "fail" for f in families.values())
                      else "pass")
    cyber_verdict = ("fail" if "fail" in (poison["verdict"], extraction["verdict"]) else "pass")
    report = {
        "system_id": SYSTEM_ID, "as_of": as_of.isoformat(), "lesson": LESSON_ID,
        "rows": int(len(TEST_LABELS)),
        "accuracy": {"headline": headline["accuracy"], "recall": headline["recall"],
                     "false_positive_rate": headline["false_positive_rate"]},
        "robustness": {"floor": {k: floor[k] for k in
                                 ("metric", "floor", "rationale", "as_of", "seq", "digest")},
                       "families": families, "verdict": robust_verdict},
        "cybersecurity": {"poisoning": poison, "extraction": extraction,
                          "verdict": cyber_verdict},
    }
    report["residual_risk"] = residual_risk(report)
    report["verdict"] = "fail" if "fail" in (robust_verdict, cyber_verdict) else "pass"
    _REPORT_CACHE["report"] = report
    return report


def _show_report() -> None:
    report = article_15_report()
    acc = report["accuracy"]["headline"]
    print(f"Article 15 evidence · {report['system_id']} · {report['as_of']} · "
          f"{report['rows']} held-out applications\n")
    print(f"  ACCURACY     declared {acc['declared']:.4f} (point {acc['point']:.4f}, "
          f"{acc['level']:.0%} CI width {acc['width']:.4f}, {acc['method']})")
    for family, cell in report["robustness"]["families"].items():
        f = cell["floor"]
        print(f"  ROBUSTNESS   {family:15s} crosses {f['floor']:.2f} at "
              f"{f['crossing_severity']}, expects {f['expected_operating_severity']:g} "
              f"-> {f['verdict'].upper()}")
    poison, extraction = report["cybersecurity"]["poisoning"], report["cybersecurity"]["extraction"]
    print(f"  POISONING    {len(poison['findings'])} finding(s) {poison['findings']}, "
          f"{len(poison['insufficient'])} batch(es) below the {poison['min_support']}-row floor")
    print(f"  EXTRACTION   {extraction['queries_to_extract']['lo']} queries on the lucky "
          f"reading -> {extraction['days_to_extract']['lo']:.2f} days at the enforced limit")
    print(f"\n  RESIDUAL     {report['residual_risk']['n_supported']} of "
          f"{len(report['residual_risk']['statements'])} risks bounded by evidence")
    for statement in report["residual_risk"]["statements"]:
        if statement["supported"]:
            b = statement["bound"]
            print(f"    bounded   {statement['id']:38s} [{b['lo']:.4f}, {b['hi']:.4f}]")
        else:
            print(f"    UNBOUNDED {statement['id']:38s} {statement['because'][:60]}")
    print(f"\n  OVERALL      {report['verdict'].upper()}")


_try("the artefact", _show_report, needs=_FOR_REPORT)

In [ ]:
def _show_audit() -> None:
    report = article_15_report()
    naked = unbounded_numbers(report)
    print(f"audit of the report: {len(naked)} numeric leaf/leaves without an interval "
          f"{naked}\n")
    typical = {"accuracy": 0.89, "recall": 0.90, "false_positive_rate": 0.12,
               "robustness": {"worst_case_accuracy": 0.755, "families_tested": 4},
               "cybersecurity": {"poisoned_batches_found": 1, "extraction_queries": 200},
               "rows": 1200}
    typical_naked = unbounded_numbers(typical)
    print("the same report in the shape packs are usually written in:")
    print(f"  {len(typical_naked)} of its numbers carry no uncertainty at all")
    for p in typical_naked:
        print(f"    {p}")
    print(f"\nBoth documents say the system is about {typical['accuracy']:.2f} accurate. Only "
          "one of them can be\nargued with, and the difference between them is "
          f"{len(typical_naked)} numbers and a method.")


_try("audit demo", _show_audit, needs=_FOR_REPORT + ("exercise 9",))

## 15. Common mistakes

- **Declaring the point.** A bare `accuracy:` line with one number after it is the most
  common line in an AI Act pack and the least useful. It cannot be falsified, so it cannot
  be defended either.
- **Resampling labels and predictions separately.** It looks conservative — the interval gets
  wider — and it is not conservative, it is wrong. The pairing demo measures how wrong.
- **Declaring the flattering end.** The lower bound for accuracy and the upper bound for a
  false positive rate. Using `lo` for both publishes your best case twice.
- **Choosing the floor after the sweep.** The reason this lesson has a ledger. Nobody writes
  "we picked 0.80 once we saw 0.83"; they write "0.80 reflects operating conditions".
- **Re-sealing a floor.** Same cheat, one entry later, and it is why `seal_floor` refuses a
  second floor for a metric that already has one.
- **Crossing on the point.** You declared the bound. The claim fails when the bound fails.
- **Comparing severities across families.** A 0.20 share of rows and a 0.20 multiple of a
  standard deviation are different things wearing the same number. Print the unit.
- **Accusing a 20-row batch.** Half of twenty labels flipped is the loudest rate in the table
  and the least examinable. The support floor comes before any comparison.
- **Leaving a batch in its own baseline.** With the suspect batch inside it, the baseline rises
  towards the suspect and the comparison quietly loses power.
- **Claiming a rate limit is a control without measuring the budget.** Until you know how many
  queries the model is worth, a rate limit is a setting.
- **Writing the residual risk last, in prose.** The three unbounded risks in section 12 are the
  ones a generated statement keeps and a written one loses.

Before you answer, print the figures the questions are about, so that what you are reading
is what this run measured rather than what somebody typed.

In [ ]:
def _show_self_check_figures() -> None:
    report = article_15_report()
    acc = report["accuracy"]["headline"]
    ext = report["cybersecurity"]["extraction"]
    budget = ext["queries_to_extract"]
    tiny = min(report["cybersecurity"]["poisoning"]["batches"].items(),
               key=lambda kv: kv[1]["n"])
    print(f"accuracy      point {acc['point']:.4f}  [{acc['lo']:.4f}, {acc['hi']:.4f}]  "
          f"declared {acc['declared']:.4f}  on {report['rows']} held-out rows")
    print(f"extraction    {budget['lo']} queries on the attacker-lucky bound, "
          f"{budget['hi']} on the confident one")
    print(f"rate limit    {ext['current_daily_limit']:g} a day in force; buying "
          f"{ext['target_days']} days would need {ext['recommended_daily_limit']:g} a day")
    print(f"smallest batch {tiny[0]} at {tiny[1]['n']} rows, against a support floor of "
          f"{report['cybersecurity']['poisoning']['min_support']}")


_try("self-check figures", _show_self_check_figures, needs=_FOR_REPORT)

## 16. Self-check

1. You take the accuracy point estimate the cell above printed, round it to a whole
   percentage, and write that one figure into the instructions for use. What is wrong with
   it?
   - (a) nothing; it is what you measured
   - (b) it should be rounded harder still, to avoid false precision
   - (c) a point with no interval, level, method or sample size can be neither falsified nor
         defended; the declaration is the bound the evidence supports
   - (d) accuracy is the wrong metric for a credit decision

2. The sweep finishes and the worst family bottoms out at 0.83. A colleague proposes declaring
   the floor at 0.80, "to reflect observed operating conditions". What does the ledger do?
   - (a) accepts it; 0.80 is below every measured value, so nothing is being hidden
   - (b) refuses: accuracy has already been measured, and a floor sealed afterwards describes
         the results rather than constraining them
   - (c) accepts it and marks it provisional
   - (d) accepts it as long as a rationale is supplied

3. The cell above printed the query budget on the attacker-lucky bound, the daily limit in
   force, and the limit that would buy the target number of days — a handful of queries a
   day, which is not a service. The report should say:
   - (a) set the limit to the recommended figure and call the risk controlled
   - (b) the measurement must be wrong; no model is extractable in that few queries
   - (c) the limit in force is sufficient, since the budget is more than zero
   - (d) rate limiting cannot be the control for a model this small — say so, name what the
         control has to be instead, and keep the measured number in the pack

4. Batch `b09` has 20 rows and the highest neighbour-disagreement rate in the table. The scan
   reports it as:
   - (a) insufficient support: named, with no claim made about its rate in either direction
   - (b) the worst poisoning finding in the training set
   - (c) clean, because 20 rows cannot meaningfully be poisoned
   - (d) merged into the neighbouring batch until the sample is large enough

5. The crossing severity is the first severity at which the metric's lower bound falls below
   the floor, not the first at which its point estimate does. Why?
   - (a) because the lower bound is always the smaller number, so it is the cautious choice
   - (b) because point estimates are unreliable
   - (c) because what was declared in the instructions for use was the bound, so that is the
         claim that fails when it falls through
   - (d) because Article 15 requires it

Mark them in the next cell. The key is not in this file — only a salted hash of it.

In [ ]:
_SELF_CHECK_KEY = {
    1: "a492945744ca0f4c",
    2: "1ef6e4e447683741",
    3: "afab8d4299115066",
    4: "98eeb028ec74215b",
    5: "f8b377b5f42bbebe",
}

_SELF_CHECK_HINT = {
    1: "re-read the refusals your declare() implements in section 5.",
    2: "re-run the ordering demo at the end of section 6 and read the except branch.",
    3: "read the last paragraph the extraction table prints in section 11.",
    4: "look at the two gates at the top of poison_scan's docstring, and at their order.",
    5: "ask what number went into the instructions for use in section 7.",
}


def check_self_check(answers: dict) -> None:
    """Mark your self-check answers. Pass a dict of question number -> letter.

    Example:
        >>> check_self_check({1: "a"})          # doctest: +SKIP
          q1  not 'a' — re-read the refusals your declare() implements in section 5.
        ...
    """
    right = 0
    for question in sorted(_SELF_CHECK_KEY):
        given = str(answers.get(question, "")).strip().lower()
        digest = hashlib.sha256(f"{LESSON_ID}:q{question}:{given}".encode()).hexdigest()[:16]
        if digest == _SELF_CHECK_KEY[question]:
            right += 1
            print(f"  q{question}  correct")
        elif not given:
            print(f"  q{question}  no answer given")
        else:
            print(f"  q{question}  not {given!r} — {_SELF_CHECK_HINT[question]}")
    print(f"\n{len(_SELF_CHECK_KEY)} questions, {right} right")


# Put your own letters in, then run this cell:
# check_self_check({1: "a", 2: "a", 3: "a", 4: "a", 5: "a"})

## What you built, and where it goes next

An Article 15 evidence report in which every measured number carries its uncertainty, every
threshold carries the sequence number that proves when it was chosen, and the residual-risk
statement says out loud which three risks nothing here bounds.

The pieces travel. `bootstrap_ci` is the same interval the post-market monitoring module needs
over rolling windows, where the question becomes whether this quarter's interval overlaps last
quarter's. The sealed floor is the shape every threshold in the pack should have — the human
oversight module's override-rate floor included. And the audit in section 13 is the one gate
worth running over the whole conformity pack: walk it, and name every number nobody attached
an interval to.

**Again, and finally: this is engineering, not legal advice.**

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_bootstrap),
                              ("exercise 2", _check_declare),
                              ("exercise 3", _check_seal_floor),
                              ("exercise 4", _check_curve),
                              ("exercise 5", _check_crossing),
                              ("exercise 6", _check_poison),
                              ("exercise 7", _check_extraction),
                              ("exercise 8", _check_residual),
                              ("exercise 9", _check_audit)):
            _try(_name, _check)
    _progress_board()
    # A stub nobody has reached yet is not a failure. A check that ran and came back wrong is:
    # in a script or under CI it ends this run non-zero, rather than letting a green exit code
    # paper over it. Inside a notebook kernel the board above has already said so, in a line
    # rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))